# Week 16 — Clef Support Chatbot (RAG with Qdrant)

## Introduction

This notebook builds a retrieval-augmented support chatbot over Clef's public
company-policy documents (markdown files from their GitHub repo).

Pipeline:

1. **Chunking** — `MarkdownHeaderTextSplitter` splits each policy file on its headings
2. **Embedding** — `all-MiniLM-L6-v2` (384-dim, normalised vectors)
3. **Vector store** — Qdrant (in-memory), cosine distance, file metadata stored as payload
4. **Generation** — Groq-hosted `openai/gpt-oss-120b`, answering only from the retrieved chunks and abstaining otherwise

Goal: does the pipeline retrieve the right policy file, answer faithfully with
citations, and refuse questions the documents cannot answer?

**Scope:** Evaluation is done on a hand-written Clef eval set instead.

In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient,models
from qdrant_client.models import Distance,PointStruct,VectorParams
from pathlib import Path
from groq import Groq

## Load & Chunk Documents

**Source:** a subset of Clef's public policy files (markdown) — compensation,remote work, healthcare, leave, holidays, code of conduct, equal opportunity, plus a few files used only for the abstention test.

### Header-Based Chunking

Policy documents are already organised by headings, so splitting on `#`, `##`
and `###` keeps each chunk about one topic instead of cutting mid-paragraph at a
fixed character count. `strip_headers=False` keeps the heading text inside the
chunk, since the heading is useful context for the embedding model (e.g.
"Less than 5 Years Experience").

In [4]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

### Embeddings

Each chunk gets `source`, `file_name` and `title` metadata so answers can cite the file they came from. Embeddings are L2-normalised
(`normalize_embeddings=True`), so cosine similarity in Qdrant is equivalent to a dot product and the query vectors must be normalised the same way.

In [5]:
docs_dir = Path("/content/documents")
docs = []
model = SentenceTransformer("all-MiniLM-L6-v2")
text =[]
for file_path in docs_dir.glob("*.md"):
  text_content = file_path.read_text(encoding='utf-8')
  chunks = markdown_splitter.split_text(text_content)

  for chunk in chunks:
    chunk.metadata['source'] = str(file_path)
    chunk.metadata['file_name']=file_path.name
    chunk.metadata['title']=file_path.stem
    text.append(chunk.page_content)
    docs.append(chunk)
emb = model.encode(text,show_progress_bar=True,
                   normalize_embeddings=True)
print(len(docs))
emb.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

11


(11, 384)

## Qdrant Setup & Indexing

An in-memory Qdrant collection (`clef_documents`) using `Distance.COSINE` and a vector size taken from `emb.shape[1]` (384), so the collection always matches the embedding model. Every chunk is upserted as one point with its text and file
metadata as payload. The collection is rebuilt on every run, which is fine for
11 chunks but would need a persistent or cloud Qdrant instance for a real
deployment.

In [6]:
client = QdrantClient(":memory:")
client.create_collection("clef_documents",
                  vectors_config=VectorParams(size=emb.shape[1],
                                              distance=Distance.COSINE))
client.upsert(collection_name='clef_documents',
              points=[
                  PointStruct(id=i,
                              vector=emb[i],
                              payload={
                                  "page":docs[i].page_content,
                                  "source":docs[i].metadata['source'],
                                  "file_name":docs[i].metadata['file_name'],
                                  "title":docs[i].metadata['title']
                              }) for i in range(len(docs))
              ])

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

## LLM Setup

Generation uses `openai/gpt-oss-120b` through the Groq API. The API key is
loaded from Colab Secrets (`GROQ_API_KEY`) and never written into the notebook.

In [7]:
from google.colab import userdata
key = userdata.get('GROQ_API_KEY')
llm = Groq(api_key=key)
MODEL="openai/gpt-oss-120b"

## Retrieval & Generation

- `retrieve(query, k=5)` embeds the query and returns the top-k chunks with their text, `source` and `file_name`
- `generate(query, k=5)` wraps the query in `<query>` tags and the chunks in `<context>` tags, then sends them to the LLM

The system prompt makes the bot introduce itself as **Clef Support AI**, cite the
source file, and reply with a fixed refusal phrase when the context does not
contain the answer. That fixed phrase makes abstention easy to detect during
evaluation.

In [8]:
def retrieve(query,k=5):
  hits = client.query_points(collection_name='clef_documents',
                      query = model.encode(query,normalize_embeddings=True).tolist(),
                      limit = k
                      ).points
  context = [
      {
          "page":hits[i].payload['page'],
          "source":hits[i].payload['source'],
          "file_name":hits[i].payload['file_name']
      } for i in range(len(hits))
  ]
  return context

def generate(query,k=5):
  context = retrieve(query,k)
  messages=[]
  messages.append({
    "role":"system",
    "content":'''
    #Role: You are a Company Policy Chat bot for company: Clef

    ##Task:
    You identy yourself as Clef Support AI, a bot that helps understand Clef's Company policy, when someone greets you,
    you have to greet them back.

    You also have to respond to user queries enclosed within <query></query> using the centext enclosed within
    <context> and </context>, also mention source and file_name given in it.
    Please refer to chat history for answering incrementally built queries
    ##Constraints:
    if you're not able to answer using the context reply:"Please ask questions that are in line with company policy".
    '''
  })
  messages.append({"role":"user",
                  "content": f'''
    <query>{query}</query>
    <context>{context}</context>
  '''})
  resp = llm.chat.completions.create(
      model = MODEL,
      messages=messages,
      temperature=0.9,top_p=0.9,max_tokens=1000
  )
  print(resp.choices[0].message.content)
  messages.append({
      "role":"assistant",
      "content":resp.choices[0].message.content
  })


## Evaluation Set

20 hand-written questions across 12 categories (factual retrieval, policy &
process, benefits, holidays, equity mechanics, multi-document synthesis, etc.).
Each has a `ground_truth` answer, the expected `source_file`, and the exact
`context_snippet` it should come from.

In [9]:
questions = [
  {
    "id": "eval_001",
    "category": "Factual Retrieval",
    "question": "What is the standard salary and equity offer for a technical candidate with less than 5 years of experience at Clef?",
    "ground_truth": "A technical candidate with less than 5 years of experience is offered a salary of $100k and 41,963 Clef stock options (~0.9% of outstanding shares). They also have the option to reduce their salary by $5k/year to receive an additional ~0.1% equity (4,663 options).",
    "source_file": "Salary and Equity Compensation.md",
    "source_url": "documents/Salary and Equity Compensation.md",
    "context_snippet": "Less than 5 Years Experience | Technical: $100k. Every employee will be offered 41,963 Clef stock options (~.9% of outstanding shares). The offer also comes with the option to take $5k less salary in exchange for ~.1% more equity."
  },
  {
    "id": "eval_002",
    "category": "Factual Retrieval",
    "question": "What are the standard salaries for technical and non-technical employees with more than 5 years of experience?",
    "ground_truth": "Technical employees with more than 5 years of experience receive $120k per year, while non-technical employees receive $100k per year.",
    "source_file": "Salary and Equity Compensation.md",
    "source_url": "documents/Salary and Equity Compensation.md",
    "context_snippet": "More than 5 Years Experience | Technical: $120k | Non-Technical: $100k"
  },
  {
    "id": "eval_003",
    "category": "Factual Retrieval",
    "question": "How much salary do the three Clef founders make per year?",
    "ground_truth": "The three Clef founders' salaries do not follow the standard rubric and are all $50k per year.",
    "source_file": "Salary and Equity Compensation.md",
    "source_url": "documents/Salary and Equity Compensation.md",
    "context_snippet": "Note: The three Clef founders' salaries do not follow this rubric and are all $50k per year."
  },
  {
    "id": "eval_004",
    "category": "Policy & Process",
    "question": "What steps must an employee take before starting an extended remote work period (longer than 2 days)?",
    "ground_truth": "An employee must: 1) Get manager approval at least two weeks in advance before making travel arrangements; 2) Give the team at least 7 days' notice by posting in #office with a @channel tag, speaking verbally with close teammates, and updating their calendar; 3) Ensure beforehand they have fast Wi-Fi, a distraction-free space, a private place for calls, and can work full days.",
    "source_file": "Working Remotely.md",
    "source_url": "documents/Working Remotely.md",
    "context_snippet": "You need your manager's approval before you can work remotely for an extended period of time... check-in should happen at least two weeks in advance... Give the team as much notice as possible, but at least 7 days... Post in #office with a @channel tag..."
  },
  {
    "id": "eval_005",
    "category": "Policy & Process",
    "question": "What defines an extended remote work period at Clef?",
    "ground_truth": "An extended remote work period includes anything longer than 2 consecutive days or any period of time where an employee works from somewhere irregular (such as a partner's home in a different city).",
    "source_file": "Working Remotely.md",
    "source_url": "documents/Working Remotely.md",
    "context_snippet": "An extended remote work period includes anything longer than 2 days or any period of time where we work from somewhere irregular."
  },
  {
    "id": "eval_006",
    "category": "Policy & Process",
    "question": "What meeting must be scheduled following an extended remote work period?",
    "ground_truth": "Before working remotely for an extended period, an employee should schedule a dedicated retrospective with their manager for when they return to debrief what worked and didn't work.",
    "source_file": "Working Remotely.md",
    "source_url": "documents/Working Remotely.md",
    "context_snippet": "Before an employee works remotely for an extended period of time (defined below), they should schedule a retrospective with their manager for when they get back."
  },
  {
    "id": "eval_007",
    "category": "Benefits & Accrual",
    "question": "How much paid vacation and sick leave do employees accrue per year, and what are the accrual caps?",
    "ground_truth": "Employees accrue 3 weeks (15 days) of paid vacation per year, accruing at 1.25 days per month. Sick leave accrues at 1 hour for every 30 hours worked, capped at a maximum of 5 days of sick leave.",
    "source_file": "Vacation and Sick Leave.md",
    "source_url": "documents/Vacation and Sick Leave.md",
    "context_snippet": "Clef offers 3 weeks (15 days) of paid vacation every year that accrues 1.25 of a day per month of work... Employees also accrue 1 hour of sick leave for every 30 hours of work, but cannot accrue more than 5 days of sick leave."
  },
  {
    "id": "eval_008",
    "category": "Benefits & Medical",
    "question": "What percentage of healthcare coverage costs does Clef contribute for employees and their dependents?",
    "ground_truth": "Clef contributes 85% of the cost for all employee baseline policies and 50% for dependents. Employees who choose to waive medical coverage receive a medical waiver allowance of $25 per month.",
    "source_file": "Healthcare and Disability Insurance.md",
    "source_url": "documents/Healthcare and Disability Insurance.md",
    "context_snippet": "Clef has chosen a baseline plan for which it contributes 85% of the cost for all employee policies and 50% for dependents... Clef offers a medical waiver allowance of $25 per month for employees who have other medical coverage and choose to waive medical coverage."
  },
  {
    "id": "eval_009",
    "category": "Benefits & Medical",
    "question": "When do new employees become eligible for medical, dental, and vision coverage at Clef?",
    "ground_truth": "All employees are eligible for healthcare coverage on the first of the month following their hire date.",
    "source_file": "Healthcare and Disability Insurance.md",
    "source_url": "documents/Healthcare and Disability Insurance.md",
    "context_snippet": "All employees are eligible for coverage on the first of the month following their hire date."
  },
  {
    "id": "eval_010",
    "category": "Equity Mechanics",
    "question": "What is the vesting schedule for stock options granted to new employees at Clef?",
    "ground_truth": "Clef stock options vest over 6 years with a 1-year cliff.",
    "source_file": "Salary and Equity Compensation.md",
    "source_url": "documents/Salary and Equity Compensation.md",
    "context_snippet": "Employee equity vests over 6 years with a 1 year cliff (while 4 years with a 1 year cliff is standard)."
  },
  {
    "id": "eval_011",
    "category": "Community & Conduct",
    "question": "Who should an employee contact immediately if they experience or witness harassment in the Clef community?",
    "ground_truth": "They should contact B immediately at b@getclef.com or reach out to one of the other founders.",
    "source_file": "Code of Conduct in the Community.md",
    "source_url": "documents/Code of Conduct in the Community.md",
    "context_snippet": "If you are being harassed, notice that someone else is being harassed, or have any other concerns, please contact B ( b@getclef.com ) or one of the other founders immediately."
  },
  {
    "id": "eval_012",
    "category": "Community & Conduct",
    "question": "What platforms and spaces are governed by Clef's Code of Conduct?",
    "ground_truth": "It applies to community and team members in all Clef online communities (IRC, support chat, Slack, Twitter, Facebook), in the Clef office, and at all Clef-hosted events.",
    "source_file": "Code of Conduct in the Community.md",
    "source_url": "documents/Code of Conduct in the Community.md",
    "context_snippet": "This code of conduct applies to community members and Clef team members in all Clef communities online (including but not limited to IRC, support chat, the Clef Community Slack group, Twitter, and Facebook), in the Clef office, and at all events hosted by Clef."
  },
  {
    "id": "eval_013",
    "category": "Holidays & Time Off",
    "question": "How are corporate holidays handled if they land on a weekend?",
    "ground_truth": "If a holiday falls on a Saturday, it is usually observed on the preceding Friday. If it falls on a Sunday, it is observed on the following Monday.",
    "source_file": "Holiday List.md",
    "source_url": "documents/Holiday List.md",
    "context_snippet": "Usually, if a holiday falls on a Saturday, we will observe it on the Friday before. If it falls on a Sunday, it will be observed on the following Monday."
  },
  {
    "id": "eval_014",
    "category": "Holidays & Time Off",
    "question": "Which holidays are observed in November by Clef?",
    "ground_truth": "Veterans Day (November 11) and Thanksgiving Day and Friday (Fourth Thursday in November and the following Friday).",
    "source_file": "Holiday List.md",
    "source_url": "documents/Holiday List.md",
    "context_snippet": "| Veterans Day | November 11 |\n| Thanksgiving Day and Friday | Fourth Thursday in November and following Friday |"
  },
  {
    "id": "eval_015",
    "category": "Equal Opportunity",
    "question": "Does Clef accommodate employees with disabilities during hiring or employment?",
    "ground_truth": "Yes, when necessary, Clef will reasonably accommodate employees and applicants with disabilities if the person is otherwise qualified to safely perform essential job functions.",
    "source_file": "Equal Opportunity Employment.md",
    "source_url": "documents/Equal Opportunity Employment.md",
    "context_snippet": "When necessary, the company will reasonably accommodate employees and applicants with disabilities if the person is otherwise qualified to safely perform all of the essential functions of the position."
  },
  {
    "id": "eval_016",
    "category": "Subsidies & Expenses",
    "question": "Does Clef cover co-working space costs during extended remote trips?",
    "ground_truth": "Yes, while Clef is on its seed extension, Clef will subsidize co-working space costs up to $10/day. This policy will be revisited once Clef reaches profitability or raises its next major round.",
    "source_file": "Working Remotely.md",
    "source_url": "documents/Working Remotely.md",
    "context_snippet": "While Clef is on its seed extension, Clef will subsidize co-working space costs for up to $10/day. Once Clef reaches profitability or raises its next major fundraising round, we will revisit this subsidy."
  },
  {
    "id": "eval_017",
    "category": "Insurance & Disability",
    "question": "What insurance coverage options does Clef provide beyond basic health insurance?",
    "ground_truth": "Clef provides Short Term Disability Insurance, Long Term Disability Insurance, and Death and Dismemberment Insurance coverage at one times earnings (with age restrictions starting at age 65).",
    "source_file": "Healthcare and Disability Insurance.md",
    "source_url": "documents/Healthcare and Disability Insurance.md",
    "context_snippet": "All employees receive Short Term Disability Insurance and Long Term Disability Insurance. All employees are also covered by Death and Dismemberment Insurance coverage at one times earnings..."
  },
  {
    "id": "eval_018",
    "category": "Reporting & Administration",
    "question": "Who must employees notify when taking vacation or sick days, and how is it tracked?",
    "ground_truth": "Employees must report vacation and sick days to the founder they report to, who records it in the payroll system to be displayed on pay stubs.",
    "source_file": "Vacation and Sick Leave.md",
    "source_url": "documents/Vacation and Sick Leave.md",
    "context_snippet": "Employees should report vacation and sick days to the founder they report to, who will mark it in the payroll system (which keeps track of accrued days and should include them on every pay stub)."
  },
  {
    "id": "eval_019",
    "category": "Multi-Document Synthesis",
    "question": "How far in advance must employees notify their team when taking vacation versus working remotely for an extended period?",
    "ground_truth": "Vacation requests require scheduling and shared calendar updates at least 1 week in advance. Extended remote work requires manager approval 2 weeks in advance and team notification at least 7 days in advance.",
    "source_file": "Vacation and Sick Leave.md, Working Remotely.md",
    "source_url": "documents/Vacation and Sick Leave.md",
    "context_snippet": "Vacation: 'add it to their shared work calendar at least a week in advance.' Extended Remote Work: 'check-in should happen at least two weeks in advance... Give the team as much notice as possible, but at least 7 days.'"
  },
  {
    "id": "eval_020",
    "category": "Multi-Document Synthesis",
    "question": "What options do non-technical candidates with less than 5 years of experience have regarding their salary and equity package?",
    "ground_truth": "A non-technical employee with under 5 years of experience earns $70k in salary and 41,963 stock options (~0.9% equity). They can trade $5k of salary ($65k net salary) for 4,663 extra options (~0.1% equity, totaling ~1%).",
    "source_file": "Salary and Equity Compensation.md",
    "source_url": "documents/Salary and Equity Compensation.md",
    "context_snippet": "Less than 5 Years Experience | Non-Technical: $70k. Every employee will be offered 41,963 Clef stock options (~.9%). The offer also comes with the option to take $5k less salary in exchange for ~.1% more equity."
  }
]

## Retrieval Evaluation

Metrics are computed at the **file level**: a retrieval counts as a hit if any of
the top-5 chunks comes from the expected source file.

- **Hit-Rate@5** — fraction of questions where the correct file appears in the top 5
- **Precision@5** — fraction of the top 5 chunks that come from the correct file
- **MRR@5** — mean of 1 / rank of the first correct chunk


In [10]:
def evaluate_retrieval(questions,k=5):
  hits=0
  MRR =0
  precision = 0
  total_questions = len(questions)
  for question in questions:
    context = retrieve(question['question'])
    files = [file['file_name'] for file in context]
    if question['source_file'] in files:
      hits +=1
      precision += (files.count(question['source_file']))/len(context)
      MRR += 1/(files.index(question['source_file'])+1)
  print("|| Hit_Rate@5 ||Precision@5||MRR@5||")
  print("="*30)
  print("||",round(hits/total_questions,2),"||",round(precision/total_questions,2),"||",round(MRR/total_questions,2),"||")

In [11]:
evaluate_retrieval(questions)

|| Hit_Rate@5 ||Precision@5||MRR@5||
|| 0.95 || 0.31 || 0.87 ||


In [13]:
evaluate_retrieval([questions[15]])

|| Hit_Rate@5 ||Precision@5||MRR@5||
|| 1.0 || 0.6 || 1.0 ||


**Retrieval result:** Hit-Rate@5 = 0.95, Precision@5 = 0.31, MRR@5 = 0.87

| Hit-Rate@5 | Precision@5 | MRR@5 |
| - | - | - |
| 0.95 | 0.31 | 0.87 |

The correct file is retrieved for 19 of 20 questions, usually at rank 1. Precision
looks low, but it is capped by the corpus: most files have only 1–2 chunks, so
even a perfect retriever cannot fill 5 slots with chunks from one file. The
single-question check above (3 of 5 chunks correct) is a file with several chunks.

## Abstention Test

5 questions that look policy-related but ask for details the documents do not
contain (e.g. a BAC limit, a room number, encrypted flash drives). The correct
behaviour is the fixed refusal phrase, not an invented answer.

In [14]:
Abstent_q=[
  {
    "id": 1,
    "type": "unanswerable",
    "question": "According to Clef's Complaint Policy, what is the exact room number of the office where founders hold confidential disciplinary hearings?",
    "expected_behavior": "abstain",
    "ground_truth_reason": "The Complaint Policy specifies that founders meet to handle complaints, but it does not mention a specific office or room number.",
    "source_document": "Complaint Policy.md"
  },
  {
    "id": 2,
    "type": "unanswerable",
    "question": "What specific blood alcohol concentration (BAC) limit does Clef enforce for employees consuming beer or wine at office milestone celebrations?",
    "expected_behavior": "abstain",
    "ground_truth_reason": "The Drug and Alcohol Policy states that employees are responsible for ensuring their consumption does not make others feel uncomfortable or unsafe, but it specifies no numerical BAC threshold.",
    "source_document": "Drug and Alcohol Policy.md"
  },
  {
    "id": 3,
    "type": "unanswerable",
    "question": "Under Clef's Employee Privacy Policy, what is the official policy regarding personal encrypted flash drives brought into the workspace?",
    "expected_behavior": "abstain",
    "ground_truth_reason": "The Employee Privacy Policy covers desks, laptops, devices, keys/locks, email, and internet, but makes no reference to encrypted flash drives or personal storage drives.",
    "source_document": "Employee Privacy.md"
  },
  {
  "id": 4,
  "type": "unanswerable",
  "question": "What is the policy regarding sabbatical accrual while an employee is taking New Parent Leave?",
  "expected_behavior": "answer",
  "expected_answer": "Time spent on New Parent Leave does not accrue time towards a sabbatical, meaning the remaining time required to be eligible for a sabbatical pauses during the leave and resumes upon return.",
  "ground_truth_reason": "The New Parent Leave policy states that paid time off of any kind, including New Parent Leave, does not accrue time towards a sabbatical. For example, an employee needing 1 more year for sabbatical who takes 12 weeks of leave will still need 1 full year of work upon return.",
  "source_document": "New Parent Leave.md"
},
  {
    "id": 5,
    "type": "unanswerable",
    "question": "According to Clef's Employee Privacy Policy, what specific fine or penalty fee is charged to employees who fail to provide a copy of their desk lock key or combination?",
    "expected_behavior": "abstain",
    "ground_truth_reason": "The Employee Privacy Policy requires employees to provide a copy of the key or combination if using a lock on company property, but it mentions no fine or monetary penalty fee for non-compliance.",
    "source_document": "Employee Privacy.md"
  }
]

In [15]:
for idx,question in enumerate(Abstent_q):
  print(idx)
  generate(question['question'])

0
Please ask questions that are in line with company policy.
1
Please ask questions that are in line with company policy.
2
Please ask questions that are in line with company policy.
3
Please ask questions that are in line with company policy.
4
Please ask questions that are in line with company policy.


**Abstention result:** 5 / 5 — the bot returned the refusal phrase on every
unanswerable question and invented nothing.

## Generation Evaluation

All 20 questions were run end to end and the answers scored by hand against the
ground truth and the retrieved chunks, each on a 0–2 scale:

- **Faithfulness** — every claim is supported by the retrieved context
- **Relevance** — the answer addresses the question asked
- **Correctness** — facts and numbers match the ground truth

In [20]:
import time
latency = []
for idx,question in enumerate(questions):
  start_time = time.time()
  print(f"Question {idx+1} : {question["question"]}")
  print(f"Ground Truth : {question['ground_truth']}")
  print(f"Source: {question['source_url']}")
  print("RAG response: ")
  latency.append(time.time() - start_time)
  generate(question["question"])
print(sum(latency)/len(questions))

Question 1 : What is the standard salary and equity offer for a technical candidate with less than 5 years of experience at Clef?
Ground Truth : A technical candidate with less than 5 years of experience is offered a salary of $100k and 41,963 Clef stock options (~0.9% of outstanding shares). They also have the option to reduce their salary by $5k/year to receive an additional ~0.1% equity (4,663 options).
Source: documents/Salary and Equity Compensation.md
RAG response: 
**Clef Support AI:**  

For a technical candidate with **less than 5 years of experience**, the standard compensation package is:

| Component | Standard Offer |
|-----------|----------------|
| **Salary** | **$100,000** per year |
| **Equity** | **41,963 stock options** (approximately 0.9 % of the company’s outstanding shares) |
| **Optional swap** | The employee may elect to reduce the salary by **$5,000** in exchange for an additional **4,663 options** (about 0.1 % more equity, bringing the total to roughly 1 % of 

**Generation result:** Faithfulness = 1.9/2, Relevance = 1.9/2, Correctness = 1.85/2

| Faithfulness | Relevance | Correctness |
| - | - | - |
| 1.9/2 | 1.9/2 | 1.85/2 |

## Results Summary

| Metric | Value | Notes |
| - | - | - |
| Hit-Rate@5 | 95% | File-level, 20 questions |
| Precision@5 | 31% | Capped by 1–2 chunks per file |
| MRR@5 | 0.87 | Correct file usually ranked first |
| Avg faithfulness | 1.9/2 | Hand-scored |
| Answer relevance | 1.9/2 | Hand-scored |
| Correctness | 1.85/2 | Hand-scored |
| Correct abstention | 5 / 5 | Fixed refusal phrase |
| Avg end-to-end latency | X s | Fill in after the timing fix below |

### Insights

Retrieval is strong for a small corpus because header-based chunks line up with
the topic of each question. Generation stays grounded in the retrieved text and
cites the source file. The system prompt's fixed refusal phrase made abstention
reliable. The main limits are the tiny corpus (11 chunks, so the metrics are
generous), file-level evaluation (it does not check that the *right passage* was
retrieved), and hand-scoring on 20 questions, which is a small sample. The next
step is adding the remaining Clef policy files and re-running the same eval set.